# Week 04 - SVMs and Model Tuning

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 05  
**Estimated time:** 2-3 hours

This notebook is self-contained. You will use the Digits dataset to study SVM kernels, hyperparameter tuning, and the difference between validation performance and final test performance.


## What You Are Building

This week has four required functions:

1. `split_scale_digits(X, y, test_size, random_state)` - stratified split and scaling without leakage.
2. `evaluate_classifiers(models, X_train, X_test, y_train, y_test)` - compare classifiers on a held-out test set.
3. `manual_grid_search(param_grid, X_train, y_train, X_val, y_val)` - inspect how `C` and `gamma` affect RBF SVM validation accuracy.
4. `run_grid_search(estimator, param_grid, X_train, y_train, cv)` - use cross-validation for tuning.

Tuning is itself part of the learning process. The test set should be used for final comparison, not for repeatedly choosing hyperparameters.

Each task ends with a **Tools for this task** list naming the library functions it needs, and what each one gives you. Working out the order to put them in is the exercise. The full reference lives on the [Library Reference page](https://hugoiscracked.github.io/dmml-public/libraries.html).


In [ ]:
# Imports - keep this cell stable
import warnings
from itertools import product
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
TEST_SIZE = 0.25


## Dataset

Digits contains 8-by-8 grayscale images of handwritten digits. The feature matrix is flattened to 64 pixel-intensity columns.


In [ ]:
digits = load_digits()
X = digits.data
y = digits.target
images = digits.images
class_names = [str(i) for i in digits.target_names]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(10, 5))
for ax, image, label in zip(axes.ravel(), images[:32], y[:32]):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(str(label), fontsize=9)
    ax.axis("off")
plt.suptitle("Sample digit images")
plt.tight_layout()
plt.show()


## Task 1 - Split and Scale Digits

Implement `split_scale_digits(X, y, test_size, random_state)`.

Rules:

- Use a stratified train/test split.
- Fit `StandardScaler` only on the training set.
- Transform train and test features with the fitted scaler.
- Return `(X_train_scaled, X_test_scaled, y_train, y_test, scaler)`.

**Tools for this task** — reference: [scikit-learn](https://hugoiscracked.github.io/dmml-public/libraries.html#sklearn)

- `train_test_split(..., stratify=y, random_state=...)` — stratified so every digit appears in both splits
- `scaler.fit_transform(X_train) / scaler.transform(X_test)` — fit on train only


In [ ]:
def split_scale_digits(
    X,
    y,
    test_size: float = TEST_SIZE,
    random_state: int = RANDOM_STATE,
):
    """Create a stratified split and scale features without leakage."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 1
X_train, X_test, y_train, y_test, scaler = split_scale_digits(X, y)

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]
assert X_train.shape[1] == X.shape[1]
assert hasattr(scaler, "mean_"), "Return the fitted scaler"
assert set(np.unique(y_train)) == set(np.unique(y))
assert set(np.unique(y_test)) == set(np.unique(y))
assert np.allclose(X_train.mean(axis=0), 0, atol=1e-7)

print("Task 1 passed")


## Task 2 - Evaluate Classifiers

Implement `evaluate_classifiers(models, X_train, X_test, y_train, y_test)`.

Input: a dictionary mapping model names to sklearn estimators.  
Output columns:

- `model`
- `accuracy`
- `f1_macro`
- `fit_time_sec`

Sort by `f1_macro` descending.

**Tools for this task** — reference: [scikit-learn](https://hugoiscracked.github.io/dmml-public/libraries.html#sklearn) · [pandas](https://hugoiscracked.github.io/dmml-public/libraries.html#pandas) · [Python](https://hugoiscracked.github.io/dmml-public/libraries.html#stdlib)

- `models.items()` — iterate name and estimator
- `model.fit(...) / .predict(...)` — the estimator contract, identical for every model here
- `accuracy_score / f1_score(average='macro')` — the two metrics
- `perf_counter()` — fit time
- `pd.DataFrame(rows).sort_values('f1_macro', ascending=False)` — assemble and rank


In [ ]:
def evaluate_classifiers(models: dict, X_train, X_test, y_train, y_test) -> pd.DataFrame:
    """Fit classifiers and return comparable held-out test metrics."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 2
baseline_models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "knn_5": KNeighborsClassifier(n_neighbors=5),
    "linear_svm": LinearSVC(max_iter=5000, random_state=RANDOM_STATE),
    "rbf_svm_default": SVC(kernel="rbf", C=1.0, gamma="scale"),
}

baseline_results = evaluate_classifiers(baseline_models, X_train, X_test, y_train, y_test)

assert isinstance(baseline_results, pd.DataFrame)
assert list(baseline_results.columns) == ["model", "accuracy", "f1_macro", "fit_time_sec"]
assert len(baseline_results) == len(baseline_models)
assert baseline_results["f1_macro"].is_monotonic_decreasing
assert baseline_results[["accuracy", "f1_macro", "fit_time_sec"]].notna().all().all()
assert baseline_results.loc[baseline_results["model"] == "rbf_svm_default", "accuracy"].iloc[0] > 0.9

print("Task 2 passed")
baseline_results


## Guided Analysis: Confusion Matrix

Use the default RBF SVM as a strong baseline and inspect where it makes mistakes.


In [ ]:
rbf_default = SVC(kernel="rbf", C=1.0, gamma="scale")
rbf_default.fit(X_train, y_train)
y_pred_default = rbf_default.predict(X_test)

cm = confusion_matrix(y_test, y_pred_default)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Default RBF SVM confusion matrix")
plt.tight_layout()
plt.show()


## Task 3 - Manual Grid Search

Implement `manual_grid_search(param_grid, X_train, y_train, X_val, y_val)`.

Rules:

- `param_grid` is a dictionary with keys `C` and `gamma`.
- Fit `SVC(kernel="rbf", C=C, gamma=gamma)` for every combination.
- Evaluate on the validation set.
- Return a dataframe with columns:
  - `C`
  - `gamma`
  - `accuracy`
  - `f1_macro`

Sort by `f1_macro` descending.

**Tools for this task** — reference: [Python](https://hugoiscracked.github.io/dmml-public/libraries.html#stdlib) · [scikit-learn](https://hugoiscracked.github.io/dmml-public/libraries.html#sklearn) · [pandas](https://hugoiscracked.github.io/dmml-public/libraries.html#pandas)

- `product(param_grid['C'], param_grid['gamma'])` — from `itertools` — every combination of the two lists
- `SVC(kernel='rbf', C=c, gamma=g)` — one model per combination
- `model.fit(X_train, y_train) / .predict(X_val)` — train on the tuning split, score on the validation split
- `accuracy_score / f1_score(average='macro')` — the two metrics
- `pd.DataFrame(rows).sort_values('f1_macro', ascending=False)` — best combination first


In [ ]:
def manual_grid_search(param_grid: dict, X_train, y_train, X_val, y_val) -> pd.DataFrame:
    """Manually evaluate RBF SVM hyperparameter combinations on a validation set."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Create a validation split from the training data for manual tuning.
X_tune, X_val, y_tune, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train,
)

svm_param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, "scale", "auto"],
}

manual_results = manual_grid_search(svm_param_grid, X_tune, y_tune, X_val, y_val)

assert isinstance(manual_results, pd.DataFrame)
assert list(manual_results.columns) == ["C", "gamma", "accuracy", "f1_macro"]
assert len(manual_results) == len(svm_param_grid["C"]) * len(svm_param_grid["gamma"])
assert manual_results["f1_macro"].is_monotonic_decreasing
assert manual_results["accuracy"].between(0, 1).all()

print("Task 3 passed")
manual_results.head()


In [ ]:
manual_pivot = manual_results.pivot(index="C", columns="gamma", values="f1_macro")
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(manual_pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
ax.set_title("Manual validation macro-F1 for RBF SVM")
plt.tight_layout()
plt.show()


## Task 4 - Cross-Validated Grid Search

Implement `run_grid_search(estimator, param_grid, X_train, y_train, cv)`.

Return `(search, cv_results)` where:

- `search` is the fitted `GridSearchCV` object.
- `cv_results` is a dataframe with columns `params`, `mean_test_score`, `std_test_score`, sorted by `mean_test_score` descending.

Use `scoring="f1_macro"`.

**Tools for this task** — reference: [scikit-learn](https://hugoiscracked.github.io/dmml-public/libraries.html#sklearn) · [pandas](https://hugoiscracked.github.io/dmml-public/libraries.html#pandas)

- `GridSearchCV(estimator, param_grid, cv=cv, scoring='f1_macro')` — the search object
- `search.fit(X_train, y_train)` — runs every combination through cross-validation
- `search.cv_results_` — a dict of arrays — wrap it in `pd.DataFrame` to get a table
- `search.best_params_ / .best_score_ / .best_estimator_` — what the search found, already refitted
- `df.sort_values('mean_test_score', ascending=False)` — the required ordering


In [ ]:
def run_grid_search(estimator, param_grid: dict, X_train, y_train, cv=5):
    """Run GridSearchCV and return the fitted search plus a compact results table."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 4
search, cv_results = run_grid_search(
    SVC(kernel="rbf"),
    svm_param_grid,
    X_train,
    y_train,
    cv=5,
)

assert hasattr(search, "best_estimator_")
assert isinstance(cv_results, pd.DataFrame)
assert list(cv_results.columns) == ["params", "mean_test_score", "std_test_score"]
assert len(cv_results) == len(svm_param_grid["C"]) * len(svm_param_grid["gamma"])
assert cv_results["mean_test_score"].is_monotonic_decreasing
assert search.best_score_ > 0.9

print("Task 4 passed")
print("Best params:", search.best_params_)
print("Best CV macro-F1:", search.best_score_)
cv_results.head()


In [ ]:
# Evaluate the tuned SVM on the untouched test set and assemble the results table.
tuned_svm = search.best_estimator_
tuned_pred = tuned_svm.predict(X_test)

tuned_results = pd.DataFrame([{
    "model": "rbf_svm_tuned_cv",
    "accuracy": accuracy_score(y_test, tuned_pred),
    "f1_macro": f1_score(y_test, tuned_pred, average="macro"),
    # GridSearchCV total timing is not tracked by run_grid_search; keep the metric table numeric.
    "fit_time_sec": np.nan,
}])

all_results = pd.concat([baseline_results, tuned_results], ignore_index=True)
all_results = all_results.sort_values("f1_macro", ascending=False).reset_index(drop=True)
all_results


## Guided Analysis: Support Vectors

Support vectors are the training points that define the SVM boundary. For a high-dimensional dataset, we can inspect them through a 2D projection.


In [ ]:
# Visualise support vectors using the first two principal components.
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(X_train)

support_mask = np.zeros(len(X_train), dtype=bool)
support_mask[tuned_svm.support_] = True

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap="tab10", s=20, alpha=0.35)
ax.scatter(X_train_2d[support_mask, 0], X_train_2d[support_mask, 1], facecolors="none", edgecolors="black", s=65, linewidths=0.8)
ax.set_title("Tuned RBF SVM support vectors in PCA space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

print("Number of support vectors:", len(tuned_svm.support_))


## Save to the Course Benchmark

Every week writes its scores into one shared table, `common/benchmark_results.csv`.
Run this cell once your numbers are final, then open `../common/benchmark.ipynb`
to see this week next to everything you have run so far.

The row to watch: how much did tuning actually buy you over the default RBF SVM?


In [ ]:
# Save this week's results to the shared, course-long benchmark table.
import sys
sys.path.append("../common")
from dmml_benchmark import save_results

save_results(
    all_results,
    week="W04",
    dataset="Digits",
    task_type="classification",
    target="digit",
    split="75_25_stratified_random_state_42",
    metrics=["accuracy", "f1_macro"],
    notes="baselines plus RBF SVM tuned by 5-fold cross-validation",
)


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - PCA Before SVM
Apply PCA with 20 components before the SVM. Add the resulting model to the shared benchmark with a second `save_results` call.

### Track B - Kernel Comparison
Add polynomial SVM variants with degree 2 and degree 3 to the benchmark.

### Track C - Tuning Cost
Add a small table comparing tuning time and test score for manual validation vs cross-validation.


In [ ]:
# Optional challenge workspace
# Your code here
